# Pairwise Spearman correlation analysis

Reproduces Figure A.2 using MIR-predicted EC$_{1:2}$, not the exploratory laboratory-EC alternative. Panels show global, Perkerra and Tana correlation structures.

The notebook contains only the analysis retained in the final thesis. Exploratory alternatives, superseded cells, personal paths, saved outputs, and unpublished figure-formatting trials have been removed.

In [ ]:
# ============================================================
# 2.4 CORRELATION ANALYSIS (SUPPORTING) – COMBINED FIGURE
# Spearman correlation including Ksat
# ------------------------------------------------------------
# Variables:
#   salinity, sodicity, hydraulic conductivity, soil chemistry,
#   microbiology, and macrofauna biomass*
#
# Note:
#   * Macrofauna biomass was only measured in Layer 1.
# ============================================================

import os
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from scipy.stats import spearmanr

# ------------------------------------------------------------
# 1. LOAD DATA AND DEFINE OUTPUT
# ------------------------------------------------------------
PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

file_path = PROJECT_DIR / "data" / "soil_data.xlsx"
sheet_name = "PredictionNewSamplesSalinity_Fi"
out_dir = PROJECT_DIR / "outputs" / "05_pairwise_spearman"
out_dir.mkdir(parents=True, exist_ok=True)

if not file_path.exists():
    raise FileNotFoundError(
        f"Input file not found: {file_path}\n"
        "Place the dataset in data/soil_data.xlsx or update file_path."
    )

df = pd.read_excel(file_path, sheet_name=sheet_name)

# ------------------------------------------------------------
# 3. CLEANING
# ------------------------------------------------------------
df.columns = df.columns.str.strip()

df["Zone"] = df["SITE"].astype(str).str.strip().str.upper()
df = df[df["Zone"].isin(["PERKERRA", "TANA"])].copy()

# ------------------------------------------------------------
# 4. VARIABLES
# ------------------------------------------------------------
vars_corr = [
    "EC 1:2 (dS/cm)",
    "ESP est",
    "Ksat",
    "SOC (%)",
    "pH",
    "Clay (%)",
    "CEC cmolc/kg",
    "TotalBacteria",
    "TotalFungi",
    "TYPES_BACTERIA",
    "TYPES_FUNGI",
    "TotalMacrofaunaBiomass (g)"
]

vars_corr = [v for v in vars_corr if v in df.columns]

for col in vars_corr:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# ------------------------------------------------------------
# 5. RENAME FOR PUBLICATION-STYLE LABELS
# ------------------------------------------------------------
rename_dict = {
    "EC 1:2 (dS/cm)": r"EC$_{1:2}$",
    "ESP est": "ESP",
    "Ksat": r"K$_{sat}$",
    "SOC (%)": "SOC",
    "pH": "pH",
    "Clay (%)": "Clay",
    "CEC cmolc/kg": "CEC",
    "TotalBacteria": "Bacterial biomass",
    "TotalFungi": "Fungal biomass",
    "TYPES_BACTERIA": "Bacterial richness",
    "TYPES_FUNGI": "Fungal richness",
    "TotalMacrofaunaBiomass (g)": "Macrofauna biomass*"
}

df = df.rename(columns=rename_dict)
vars_corr_clean = [rename_dict.get(v, v) for v in vars_corr]

# ------------------------------------------------------------
# 6. ORDER OF VARIABLES
# ------------------------------------------------------------
ordered_vars = [
    r"EC$_{1:2}$",
    "ESP",
    r"K$_{sat}$",
    "SOC",
    "pH",
    "Clay",
    "CEC",
    "Bacterial biomass",
    "Fungal biomass",
    "Bacterial richness",
    "Fungal richness",
    "Macrofauna biomass*"
]

ordered_vars = [v for v in ordered_vars if v in vars_corr_clean]

# ------------------------------------------------------------
# 7. SPEARMAN FUNCTION
# ------------------------------------------------------------
def spearman_corr_with_pvalues(data, variables):
    corr = pd.DataFrame(index=variables, columns=variables, dtype=float)
    pval = pd.DataFrame(index=variables, columns=variables, dtype=float)

    rows = []

    for v1 in variables:
        for v2 in variables:
            valid = data[v1].notna() & data[v2].notna()

            if valid.sum() >= 3:
                r, p = spearmanr(data.loc[valid, v1], data.loc[valid, v2])
            else:
                r, p = np.nan, np.nan

            corr.loc[v1, v2] = r
            pval.loc[v1, v2] = p

            rows.append({
                "Var1": v1,
                "Var2": v2,
                "rho": r,
                "p": p,
                "N_pair": int(valid.sum())
            })

    return corr, pval, pd.DataFrame(rows)

# ------------------------------------------------------------
# 8. BUILD ANNOTATION MATRIX
#    Show values only when |rho| >= 0.40
# ------------------------------------------------------------
def build_annotation_matrix(corr, threshold=0.40):
    annot = corr.copy().astype(object)

    for i in range(corr.shape[0]):
        for j in range(corr.shape[1]):
            val = corr.iloc[i, j]
            if pd.isna(val):
                annot.iloc[i, j] = ""
            elif abs(val) >= threshold:
                annot.iloc[i, j] = f"{val:.2f}"
            else:
                annot.iloc[i, j] = ""

    return annot

# ------------------------------------------------------------
# 9. RUN CORRELATION
# ------------------------------------------------------------
def run_corr(data, name):
    data = data[ordered_vars].copy()

    corr, pval, long = spearman_corr_with_pvalues(data, ordered_vars)

    with pd.ExcelWriter(os.path.join(out_dir, f"spearman_{name}.xlsx")) as writer:
        corr.to_excel(writer, sheet_name="rho")
        pval.to_excel(writer, sheet_name="pvalues")
        long.to_excel(writer, sheet_name="long", index=False)

    sig = long[
        (long["p"] < 0.05) &
        (long["Var1"] != long["Var2"])
    ].copy()

    sig["pair"] = sig.apply(
        lambda r: "_".join(sorted([r["Var1"], r["Var2"]])),
        axis=1
    )
    sig = sig.drop_duplicates("pair").drop(columns="pair")
    sig = sig.sort_values("rho", ascending=False)

    sig.to_excel(os.path.join(out_dir, f"significant_{name}.xlsx"), index=False)

    print(f"\n{name.upper()} DONE")
    print(sig.head(15))

    return corr, pval, sig

# ------------------------------------------------------------
# 10. COMPUTE MATRICES
# ------------------------------------------------------------
corr_global, pval_global, sig_global = run_corr(df, "global")

corr_perkerra, pval_perkerra, sig_perkerra = run_corr(
    df[df["Zone"] == "PERKERRA"], "perkerra"
)

corr_tana, pval_tana, sig_tana = run_corr(
    df[df["Zone"] == "TANA"], "tana"
)

# ------------------------------------------------------------
# 11. PLOT SINGLE COMBINED FIGURE
# ------------------------------------------------------------
sns.set_theme(style="white")
plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 14,
    "axes.titlesize": 18,
    "axes.labelsize": 14,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11
})

fig = plt.figure(figsize=(18, 17))

# Más espacio entre filas y columnas
gs = fig.add_gridspec(
    nrows=2,
    ncols=2,
    height_ratios=[1.18, 1.0],
    width_ratios=[1, 1],
    hspace=0.34,
    wspace=0.22
)

# Panel superior ocupando ambas columnas
ax_a = fig.add_subplot(gs[0, :])
ax_b = fig.add_subplot(gs[1, 0])
ax_c = fig.add_subplot(gs[1, 1])

def draw_heatmap(ax, corr, show_cbar=False):
    mask = np.triu(np.ones_like(corr, dtype=bool))
    annot_mat = build_annotation_matrix(corr, threshold=0.40)

    hm = sns.heatmap(
        corr,
        mask=mask,
        cmap="coolwarm",
        vmin=-1,
        vmax=1,
        center=0,
        linewidths=0.7,
        linecolor="white",
        square=True,
        annot=annot_mat,
        fmt="",
        annot_kws={"size": 10, "weight": "bold"},
        cbar=show_cbar,
        cbar_kws={"shrink": 0.82, "pad": 0.02},
        ax=ax
    )

    ax.set_title("")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
    ax.tick_params(axis="both", length=0)

    if show_cbar:
        cbar = hm.collections[0].colorbar
        cbar.ax.tick_params(labelsize=11)
        cbar.set_label("Spearman's rho", fontsize=13)

# Dibujar mapas
draw_heatmap(ax_a, corr_global, show_cbar=True)
draw_heatmap(ax_b, corr_perkerra, show_cbar=False)
draw_heatmap(ax_c, corr_tana, show_cbar=False)

# Reposicionar a) un poco más centrado reduciendo su ancho efectivo
pos_a = ax_a.get_position()
new_width = pos_a.width * 0.86
new_x0 = pos_a.x0 + (pos_a.width - new_width) / 2 - 0.1
ax_a.set_position([new_x0, pos_a.y0, new_width, pos_a.height])

# Bajar un poco más los paneles inferiores para evitar que tapen etiquetas del panel a)
pos_b = ax_b.get_position()
pos_c = ax_c.get_position()

shift_down = 0.018
ax_b.set_position([pos_b.x0, pos_b.y0 - shift_down, pos_b.width, pos_b.height])
ax_c.set_position([pos_c.x0, pos_c.y0 - shift_down, pos_c.width, pos_c.height])

# Panel labels fuera del gráfico
ax_a.text(-0.08, 1.05, "a)", transform=ax_a.transAxes,
          fontsize=18, fontweight="bold", va="bottom", ha="left")
ax_b.text(-0.10, 1.05, "b)", transform=ax_b.transAxes,
          fontsize=18, fontweight="bold", va="bottom", ha="left")
ax_c.text(-0.10, 1.05, "c)", transform=ax_c.transAxes,
          fontsize=18, fontweight="bold", va="bottom", ha="left")

# Guardar figura combinada
combined_path = os.path.join(out_dir, "figure_A_2_pairwise_spearman.png")
plt.savefig(combined_path, dpi=600, bbox_inches="tight")
plt.show()
